In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Downloading the dataset to google colab for better speed**

In [ ]:
#Link to train and test dataset present in google drive
!mkdir -p /content/asl_data
!cp -r "/content/drive/MyDrive/thesis/data/asl_alphabet_train/asl_alphabet_train" /content/asl_data/
!cp -r "/content/drive/MyDrive/thesis/data/asl_alphabet_test" /content/asl_data/

#Setting path variables
original_data_path = "/content/asl_data/asl_alphabet_train"
test_folder        = "/content/asl_data/asl_alphabet_test/asl_alphabet_test"

**Training SLTUNET with first half of ASL with 80-20 split. Saving the model in drive**

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


#Strong augmentation for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#Same transform for validation and training
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(original_data_path, transform=train_transform)

# Taking the first half of the dataset
total_size = len(full_dataset)
half_size = total_size // 2

first_half_indices = list(range(half_size))

#Spliting the first half into train and validation
train_size = int(0.8 * half_size)
val_size = half_size - train_size

train_indices = first_half_indices[:train_size]
val_indices   = first_half_indices[train_size:]

from torch.utils.data import Subset

train_dataset = Subset(full_dataset, train_indices)
val_dataset   = Subset(datasets.ImageFolder(original_data_path, transform=val_test_transform), val_indices)

print(f"Full dataset: {total_size} images")
print(f"First half used: {half_size} images")
print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} images")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

class_names = full_dataset.classes
num_classes = len(class_names)

#SLTUNet with Pretrained ResNet-50
class SLTUNet_Pretrained(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()
        base = models.resnet50(pretrained=True)
        self.enc1 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.enc2 = base.layer1
        self.enc3 = base.layer2
        self.enc4 = base.layer3
        self.enc5 = base.layer4
        self.reduce = nn.Conv2d(2048, 512, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.enc1(x); x = self.enc2(x); x = self.enc3(x)
        x = self.enc4(x); x = self.enc5(x); x = self.reduce(x)
        x = self.pool(x)
        return self.head(x)

model = SLTUNet_Pretrained(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
epochs = 5

best_val_acc = 0.0
save_path = "/content/drive/MyDrive/thesis/sltunet_final_best.pth"

print("Training started!")
for epoch in range(epochs):
    #Training
    model.train()
    train_correct = train_total = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Train"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, labels)
        loss.backward()
        optimizer.step()
        _, pred = outs.max(1)
        train_total += labels.size(0)
        train_correct += pred.eq(labels).sum().item()

    #validation
    model.eval()
    val_correct = val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outs = model(imgs)
            _, pred = outs.max(1)
            val_total += labels.size(0)
            val_correct += pred.eq(labels).sum().item()

    train_acc = 100. * train_correct / train_total
    val_acc = 100. * val_correct / val_total
    print(f"Epoch {epoch+1:2d} → Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")


    #Saving the best model in drive
    scheduler.step()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Best model saved! Val = {val_acc:.2f}%")

**Extended and modified test sets loading from drive.**
**Loading model from drive.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
from torchvision import models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Model path in drive
MODEL_PATH = "/content/drive/MyDrive/thesis/sltunet_final_best.pth"

#Test folder path that we wish to test
TEST_DIR   = "/content/drive/MyDrive/thesis/large_asl_test_set/test_set_8596_env_overlay" #Change the path for required modified test set
OUT_CSV    = "/content/drive/MyDrive/thesis/sltunet_test_preds.csv"

In [ ]:
class SLTUNet_Pretrained(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        self.enc1 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.enc2 = base.layer1
        self.enc3 = base.layer2
        self.enc4 = base.layer3
        self.enc5 = base.layer4

        self.reduce = nn.Conv2d(2048, 512, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.enc1(x); x = self.enc2(x); x = self.enc3(x)
        x = self.enc4(x); x = self.enc5(x); x = self.reduce(x)
        x = self.pool(x)
        return self.head(x)

In [ ]:
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

test_class_names = test_dataset.classes
print("Test classes found:", test_class_names)
print("Num test classes:", len(test_class_names))


In [ ]:
train_class_names = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','del','nothing','space'
]

model = SLTUNet_Pretrained(num_classes=len(train_class_names)).to(device)

state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded:", MODEL_PATH)


In [ ]:
from collections import defaultdict

correct = 0
total = 0

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        #Converting to class names
        pred_names = [train_class_names[p.item()] for p in preds.cpu()]
        true_names = [test_class_names[t.item()] for t in labels.cpu()]

        for pn, tn in zip(pred_names, true_names):
            total += 1
            per_class_total[tn] += 1
            if pn == tn:
                correct += 1
                per_class_correct[tn] += 1

overall_acc = 100.0 * correct / total
print(f"\n Overall Accuracy on new test set: {overall_acc:.2f}% ({correct}/{total})\n")

print("Per-class accuracy:")
for c in test_class_names:
    acc = 100.0 * per_class_correct[c] / per_class_total[c] if per_class_total[c] else 0.0
    print(f"{c:>10s}: {acc:6.2f}% ({per_class_correct[c]}/{per_class_total[c]})")


**Alphabet level analysis-Confusion matrix**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tqdm import tqdm

In [ ]:
y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Collecting predictions"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        #Converting indices to class names using training class order
        y_true.extend([test_class_names[i] for i in labels.cpu().numpy()])
        y_pred.extend([train_class_names[i] for i in preds.cpu().numpy()])

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=test_class_names
)

print("Confusion matrix shape:", cm.shape)

In [ ]:
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)

In [ ]:
plt.figure(figsize=(7, 7))
sns.heatmap(
    cm_normalized,
    xticklabels=test_class_names,
    yticklabels=test_class_names,
    cmap="Blues",
    annot=False,
    fmt=".2f"
)
plt.xlabel("Predicted Letter")
plt.ylabel("True Letter")
plt.title("Alphabet-Aware Confusion Matrix (Normalized)")
plt.tight_layout()
plt.show()